In [46]:
import pandas as pd
import numpy as np

In [1]:
import re

# 读取文件前几千行即可，通常建表语句在文件头部
with open('Chinese.sql', 'r', encoding='utf-8') as f:
    content = f.read(100000) # 读取前 100KB 内容

# 使用正则匹配 CREATE TABLE 后面跟着的名称
# 匹配格式：CREATE TABLE `tableName` 或 CREATE TABLE tableName
table_names = re.findall(r'CREATE TABLE\s+(?:IF NOT EXISTS\s+)?[`"\[]?(\w+)[`"\]]?', content, re.IGNORECASE)

print("发现的表名有:", table_names)

发现的表名有: ['xhzd_surnfu']


In [5]:
import sqlite3

# 1. 创建（或连接）一个本地数据库文件
conn = sqlite3.connect('my_data.db')
cursor = conn.cursor()

# 2. 读取 .sql 文件内容
with open('Chinese.sql', 'r', encoding='utf-8') as f:
    sql_script = f.read()

# 3. 执行脚本（构建表并插入数据）
try:
    cursor.executescript(sql_script)
    conn.commit()
    print("SQL 脚本执行成功！")
except Exception as e:
    print(f"执行出错: {e}")

# 4. 验证：读取其中一个表
import pandas as pd
# 替换 'your_table_name' 为你 sql 文件里创建的表名
df = pd.read_sql_query("SELECT * FROM xhzd_surnfu LIMIT 5", conn)
print(df)

conn.close()

执行出错: near "SET": syntax error
    id zi    py  wubi bushou  bihua pinyin  \
0  1.0  卝  guan  None    难检字    4.0   guàn   
1  2.0  羋    mi  None    难检字    8.0     mǐ   
2  3.0  羐  ling  None    难检字   10.0   líng   
3  4.0  瑴   jue  None    难检字   14.0    jué   
4  5.0  彛    yi  None    难检字   16.0     yí   

                                               jijie xiangjie  
0  卝<br>guàn<br>古代儿童将头发束成两角的样子。<br><br>卝<br>kuàng...     None  
1  羋<br>mǐ<br>同“芈”。<br><br>笔画数：8；<br>部首：难检字；<br>笔...     None  
2  羐<br>líng<br>古同“蔆”，即“菱”。<br><br>笔画数：10；<br>部首：...     None  
3  瑴<br>jué<br>玉名：“中黄瑴玉。”<br>双玉：“公为之请纳玉于王与晋侯，皆十瑴。...     None  
4  彛<br>yí<br>同“彝”。<br><br>笔画数：16；<br>部首：难检字；<br>...     None  


In [7]:
import pandas as pd
import re
import csv
from io import StringIO

file_path = 'Chinese.sql'
table_name = 'xhzd_surnfu'

all_data = []
# 你提供的 9 个标准字段
columns = ['id', 'zi', 'py', 'wubi', 'bushou', 'bihua', 'pinyin', 'jijie', 'xiangjie']

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        # 只处理目标表的插入语句
        if line.startswith(f"INSERT INTO `{table_name}`"):
            # 1. 提取 VALUES (...) 括号里的原始内容
            start = line.find("VALUES (") + 8
            end = line.rfind(");")
            raw_values = line[start:end]
            
            # 2. 使用 csv 模块解析这一行，它能自动处理引号内的逗号
            # quotechar="'" 表示单引号内的是一个整体
            reader = csv.reader([raw_values], quotechar="'", skipinitialspace=True, delimiter=',', doublequote=True)
            
            try:
                row = next(reader)
                # 3. 清洗数据：处理 null 和 多余空格
                clean_row = [None if v.strip().lower() == 'null' else v.strip() for v in row]
                
                # 检查列数是否匹配，不匹配的跳过或记录
                if len(clean_row) == len(columns):
                    all_data.append(clean_row)
                else:
                    # 如果还是不匹配，打印出来看看是哪一行有问题
                    print(f"跳过一行：预期 {len(columns)} 列，实际抓取到 {len(clean_row)} 列")
            except StopIteration:
                continue

# 转换成 DataFrame
df = pd.DataFrame(all_data, columns=columns)

# 转换数字列
df['id'] = pd.to_numeric(df['id'], errors='coerce')
df['bihua'] = pd.to_numeric(df['bihua'], errors='coerce')

print(f"成功导入 {len(df)} 条有效数据！")
print(df.head())

成功导入 20823 条有效数据！
   id zi    py  wubi bushou  bihua pinyin  \
0   1  卝  guan  None    难检字      4   guàn   
1   2  羋    mi  None    难检字      8     mǐ   
2   3  羐  ling  None    难检字     10   líng   
3   4  瑴   jue  None    难检字     14    jué   
4   5  彛    yi  None    难检字     16     yí   

                                               jijie xiangjie  
0  卝<br>guàn<br>古代儿童将头发束成两角的样子。<br><br>卝<br>kuàng...     None  
1  羋<br>mǐ<br>同“芈”。<br><br>笔画数：8；<br>部首：难检字；<br>笔...     None  
2  羐<br>líng<br>古同“蔆”，即“菱”。<br><br>笔画数：10；<br>部首：...     None  
3  瑴<br>jué<br>玉名：“中黄瑴玉。”<br>双玉：“公为之请纳玉于王与晋侯，皆十瑴。...     None  
4  彛<br>yí<br>同“彝”。<br><br>笔画数：16；<br>部首：难检字；<br>...     None  


In [9]:
df['jijie'][0]

'卝<br>guàn<br>古代儿童将头发束成两角的样子。<br><br>卝<br>kuàng<br>古同“矿”。<br><br>笔画数：4；<br>部首：难检字；<br>笔顺编号：2121'

In [10]:
import pandas as pd

# 假设 df 已经读入，列名为 'jijie'

# 1. 建立 'bishun' 字段：匹配 "笔顺编号：" 后面的所有数字
# \d+ 表示匹配一个或多个数字
df['bishun'] = df['jijie'].str.extract(r'笔顺编号：(\d+)')

# 2. 建立 '意思' 字段
# 第一步：先提取 "笔画数：" 之前的所有内容
# (.*?) 表示非贪婪匹配，直到遇到 "笔画数：" 为止
df['yisi'] = df['jijie'].str.extract(r'(.*?)笔画数：', flags=re.S) # re.S 让点号能匹配换行符

# 第二步：将 <br> 替换为空格，并去掉两端的空格/换行
df['yisi'] = df['yisi'].str.replace(r'<br\s*/?>', ' ', regex=True, case=False).str.strip()

# 查看处理后的结果
print(df.head())

   id zi    py  wubi bushou  bihua pinyin  \
0   1  卝  guan  None    难检字      4   guàn   
1   2  羋    mi  None    难检字      8     mǐ   
2   3  羐  ling  None    难检字     10   líng   
3   4  瑴   jue  None    难检字     14    jué   
4   5  彛    yi  None    难检字     16     yí   

                                               jijie xiangjie  \
0  卝<br>guàn<br>古代儿童将头发束成两角的样子。<br><br>卝<br>kuàng...     None   
1  羋<br>mǐ<br>同“芈”。<br><br>笔画数：8；<br>部首：难检字；<br>笔...     None   
2  羐<br>líng<br>古同“蔆”，即“菱”。<br><br>笔画数：10；<br>部首：...     None   
3  瑴<br>jué<br>玉名：“中黄瑴玉。”<br>双玉：“公为之请纳玉于王与晋侯，皆十瑴。...     None   
4  彛<br>yí<br>同“彝”。<br><br>笔画数：16；<br>部首：难检字；<br>...     None   

             bishun                                    yisi  
0              2121  卝 guàn 古代儿童将头发束成两角的样子。  卝 kuàng 古同“矿”。  
1          12121112                              羋 mǐ 同“芈”。  
2        2121121354                      羐 líng 古同“蔆”，即“菱”。  
3    12145111213554  瑴 jué 玉名：“中黄瑴玉。” 双玉：“公为之请纳玉于王与晋侯，皆十瑴。”  
4  5114312343453132         

In [12]:
del df['jijie']
del df['xiangjie']
df.head()

,id,zi,py,wubi,bushou,bihua,pinyin,bishun,yisi
0,1,卝,guan,None,难检字,4,guàn,2121,卝 guàn 古代儿童将头发束成两角的样子。 卝 kuàng 古同“矿”。
1,2,羋,mi,None,难检字,8,mǐ,12121112,羋 mǐ 同“芈”。
2,3,羐,ling,None,难检字,10,líng,2121121354,羐 líng 古同“蔆”，即“菱”。
3,4,瑴,jue,None,难检字,14,jué,12145111213554,瑴 jué 玉名：“中黄瑴玉。” 双玉：“公为之请纳玉于王与晋侯，皆十瑴。”
4,5,彛,yi,None,难检字,16,yí,5114312343453132,彛 yí 同“彝”。


In [55]:
df.columns=['id', 'char', 'py', 'wubi', 'bushou', 'bihua', 'pinyin', 'bishun',
       'yisi']

In [13]:
df['bushou'].unique()

array(['难检字', '丨', '亅', '丿', '乛', '一', '乙', '乚', '丶', '八', '勹', '匕', '冫',
       '卜', '厂', '刀', '刂', '儿', '二', '匚', '阝', '丷', '几', '卩', '冂', '力',
       '冖', '凵', '人', '亻', '入', '十', '厶', '亠', '匸', '讠', '廴', '又', '艹',
       '屮', '彳', '巛', '川', '辶', '寸', '大', '山', '扌', '方', '风', '父', '戈',
       '禾', '石', '虫', '言', '飞', '干', '工', '弓', '廾', '广', '己', '彐', '彑',
       '巾', '口', '马', '门', '宀', '女', '犭', '彡', '尸', '饣', '士', '氵', '纟',
       '巳', '土', '囗', '兀', '夕', '小', '忄', '骨', '幺', '弋', '尢', '夂', '子',
       '贝', '比', '灬', '长', '车', '歹', '斗', '厄', '卝', '户', '火', '旡', '见',
       '斤', '耂', '毛', '木', '肀', '牛', '牜', '爿', '片', '攴', '攵', '气', '欠',
       '犬', '日', '氏', '礻', '手', '殳', '水', '瓦', '尣', '王', '韦', '文', '毋',
       '心', '牙', '爻', '曰', '月', '爫', '支', '止', '爪', '白', '癶', '歺', '甘',
       '瓜', '钅', '立', '龙', '矛', '皿', '母', '目', '疒', '鸟', '皮', '生', '矢',
       '示', '罒', '田', '玄', '穴', '疋', '业', '衤', '用', '玉', '艸', '臣', '而',
       '耳', '缶', '艮', '虍', '臼', '耒', '米', '糸', '齐', '肉', '色', 

In [17]:
df[df['bushou']=='口']

,id,zi,py,wubi,bushou,bihua,pinyin,bishun,yisi
4068,3730,口,kou,kkkk,口,3,kǒu,251,口 kǒu 人和动物吃东西和发声的器官（亦称“嘴”）：口腔。口才。口齿。口若悬河。 容器通外...
4069,3731,叭,"ba,",kwy,口,5,"bà,bā,pā",25134,叭 bā 象声词：叭的一声，弦断了。
4070,3732,叱,chi,kxn,口,5,chì,25135,叱 chì 大声呵斥：怒叱。叱问。叱骂。叱责。叱咤（发怒的声音）。叱咤风云（形容声势威力很大）。
4071,3733,叨,"dao,tao",kvn,口,5,"dāo,tāo",25153,叨 tāo 承受：叨光。叨拢（谢人款待的话）。叨陪。 古同“饕”，贪。 叨 dāo 〔叨叨...
4072,3734,叼,diao,kngg,口,5,diāo,25151,叼 diāo 用嘴衔住：嘴里叼着烟卷。
...,...,...,...,...,...,...,...,...,...
4820,4482,囔,"nang,nang",kgke,口,25,"nāng,nɑng",2511251245251251112213534,囔 nāng 〔囔囔〕小声说话（后一个“囔”读轻声）。 〔嘟囔〕见“ 嘟”。
4821,4483,囖,luo,None,口,28,luó,2513143142522155444432411121,囖 luó 囖 luō 囖 “囉”的讹字。
20739,20740,,wai,None,口,10,wāi,NaN,NaN
20756,20757,,han,None,口,22,hǎn,NaN,NaN


In [ ]:
#爬取新华字典中的常用字2500
#次常用字 1000
#通用字 3500
#其他标注为生僻字

In [41]:
headers = {
        'User-Agent':"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0"
    }

In [42]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_common_chars(url,word_type):
    
    

    try:
        response = requests.get(url, headers=headers)
        response.encoding = response.apparent_encoding
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. 使用 find_all 获取页面中所有 class="bs_index3" 的容器
        containers = soup.find_all(class_='bs_index3')
        
        char_list = []
        
        # 2. 遍历每一个容器（每一画的容器）
        for container in containers:
            # 在当前容器下寻找所有的 <a> 标签
            char_elements = container.find_all('a')
            
            for elem in char_elements:
                char = elem.get_text().strip()
                # 3. 严格清洗：只要单个汉字
                if len(char) == 1 and '\u4e00' <= char <= '\u9fa5':
                    char_list.append(char)
        
        # 4. 转换为 DataFrame
        df = pd.DataFrame(char_list, columns=['char'])
        
        # 5. 去重（防止网页结构重叠导致抓取重复）
        df = df.drop_duplicates().reset_index(drop=True)
        
        # 6. 添加 type 列，标注为常用字
        df['type'] = word_type
        
        print(f"成功抓取到 {len(df)} 个常用字。")
        return df

    except Exception as e:
        print(f"发生错误: {e}")
        return None


In [32]:
url='https://zidian.gushici.net/t/chang2k5.html'
word_type='常用字'
        
df_base= scrape_common_chars(url,word_type)
df_base.head(3) 

成功抓取到 2500 个常用字。


,char,type
0,一,常用字
1,乙,常用字
2,二,常用字


In [33]:
url='https://zidian.gushici.net/t/chang1k.html'
word_type='次常用字'

df_l2= scrape_common_chars(url,word_type)
df_l2.head(3) 

成功抓取到 1000 个常用字。


,char,type
0,匕,次常用字
1,刁,次常用字
2,丐,次常用字


In [35]:
url='https://zidian.gushici.net/t/tong7k.html'
word_type='通用字'

df_l3= scrape_common_chars(url,word_type)
df_l3.head(3) 

成功抓取到 1310 个常用字。


,char,type
0,一,通用字
1,乙,通用字
2,二,通用字


In [47]:
# 1. 获取 A 表的集合用于快速比对
set_a = set(df_base['char'])

# 2. 合并两个表的所有不重复汉字（并集）
# 使用 set 的 union 或者 pandas 的 drop_duplicates
all_chars = pd.concat([df_base, df_l2]).drop_duplicates('char').reset_index(drop=True)

# 3. 标注分类逻辑
# 如果 汉字 在 set_a 中，标记为常用字；否则标记为通用字
all_chars['type'] = np.where(all_chars['char'].isin(set_a), '常用字', '次常用字')

# 4. 查看结果
all_chars

,char,type
0,一,常用字
1,乙,常用字
2,二,常用字
3,十,常用字
4,丁,常用字
...,...,...
3495,髓,次常用字
3496,蘸,次常用字
3497,镶,次常用字
3498,瓤,次常用字


In [48]:
# 1. 获取 A 表的集合用于快速比对
set_a = set(all_chars['char'])

# 2. 合并两个表的所有不重复汉字（并集）
# 使用 set 的 union 或者 pandas 的 drop_duplicates
all_chars = pd.concat([all_chars, df_l3]).drop_duplicates('char').reset_index(drop=True)

# 3. 标注分类逻辑
# 如果 汉字 在 set_a 中，标记为常用字；否则标记为通用字
all_chars['type'] = np.where(all_chars['char'].isin(set_a), all_chars['type'],'通用字')

# 4. 查看结果
all_chars

,char,type
0,一,常用字
1,乙,常用字
2,二,常用字
3,十,常用字
4,丁,常用字
...,...,...
3879,纰,通用字
3880,纴,通用字
3881,纶,通用字
3882,纻,通用字


In [51]:
all_chars.groupby(['type']).count()

,char
type,
常用字,2500
次常用字,1000
通用字,384


In [80]:
df.groupby('type').count()

,id,char,py,wubi,bushou,bihua,pinyin,bishun,yisi
type,,,,,,,,,
常用字,2491,2491,2491,2489,2491,2491,2491,2490,2490
次常用字,997,997,997,987,997,997,997,997,997
生僻字,16952,16952,16952,3204,16952,16952,16952,16891,16892
通用字,383,383,383,361,383,383,383,383,383


,id,char,py,wubi,bushou,bihua,pinyin,bishun,yisi,type
60,61,一,yi,ggll,一,1,yī,1,一 yī 数名，最小的正整数（在钞票和单据上常用大写“壹”代）。 纯；专：专一。一心一意。 ...,常用字


In [73]:
del df['type']
#del df['type_y']

In [77]:
df[df['type'].isna()]

,id,char,py,wubi,bushou,bihua,pinyin,bishun,yisi,type
0,1,卝,guan,None,难检字,4,guàn,2121,卝 guàn 古代儿童将头发束成两角的样子。 卝 kuàng 古同“矿”。,NaN
1,2,羋,mi,None,难检字,8,mǐ,12121112,羋 mǐ 同“芈”。,NaN
2,3,羐,ling,None,难检字,10,líng,2121121354,羐 líng 古同“蔆”，即“菱”。,NaN
3,4,瑴,jue,None,难检字,14,jué,12145111213554,瑴 jué 玉名：“中黄瑴玉。” 双玉：“公为之请纳玉于王与晋侯，皆十瑴。”,NaN
4,5,彛,yi,None,难检字,16,yí,5114312343453132,彛 yí 同“彝”。,NaN
...,...,...,...,...,...,...,...,...,...,...
20817,20818,牙合,he,None,牙,10,hē,NaN,NaN,NaN
20818,20819,乊,ho,tur,丿,3,ho,NaN,NaN,NaN
20819,20820,乥,ho lo,tunb,乙,4,ho lo,NaN,NaN,NaN
20820,20821,偾,fen,wfam,亻,11,fèn,NaN,NaN,NaN


In [78]:
#set_a = set(all_chars['char'])
#df=df.merge(all_chars,on='char',how='left')
df['type']=np.where(df['type'].isna(),'生僻字',df['type'])

In [43]:
import pandas as pd
from bs4 import BeautifulSoup

# 假设 html_content 是你获取到的网页源代码
url= "https://www.46.la/tool/primary-chinese-character-list" 

def parse_hanzi_data(url):
    response = requests.get(url, headers=headers)
    response.encoding = response.apparent_encoding
    soup = BeautifulSoup(response, 'html.parser')
    data = []

    # 1. 找到所有的标题节点
    titles = soup.find_all('h3', class_='section-title')

    for title_node in titles:
        title_text = title_node.get_text(strip=True)
        
        # 2. 找到该标题紧邻的下一个 class 为 chars-grid 的 div
        grid_node = title_node.find_next_sibling('div', class_='chars-grid')
        
        if grid_node:
            # 3. 提取 grid 内部所有的汉字项
            char_items = grid_node.find_all('div', class_='char-item')
            for item in char_items:
                word = item.get_text(strip=True)
                data.append({
                    'title': title_text,
                    'word': word
                })

    # 4. 生成 DataFrame
    df = pd.DataFrame(data)
    return df

# 使用示例
df_grade = parse_hanzi_data(url) # 如果是用 selenium
df_grade.head()

# 如果你想保存为 Excel 或 CSV
# df.to_csv("hanzi_list.csv", index=False, encoding="utf_8_sig")

ConnectTimeout: HTTPSConnectionPool(host='www.46.la', port=443): Max retries exceeded with url: /tool/primary-chinese-character-list (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000017F4327F1A0>, 'Connection to www.46.la timed out. (connect timeout=None)'))

In [83]:
#Map grade and unit:
unit_index={
    "1-上-识字1": "天地人你我他",
    "1-上-识字2": "金木水火土",
    "1-上-U4": "子人大月儿头里可东西",
    "1-上-U5": "水去来不小少牛果鸟早书本",
    "1-下-U1": "春风冬雪花飞入姓什么双",
    "1-下-U2": "吃叫主江住没以会走北京",
    "1-下-U3": "玩很当音讲行许边",
    "2-上-U1": "两哪宽顶眼睛肚皮孩跳",
    "2-上-U2": "花园桥群队旗铜号红领巾",
    "2-上-U3": "称柱底杆秤做站船然",
    "2-下-U1": "诗村童碧妆绿丝剪冲寻姑娘吐柳荡桃杏",
    "2-下-U2": "锋昨冒留弯背洒温暖能桌味买具甘甜菜劳",
    "2-下-U3": "州湾岛峡民族谊齐奋贴街舟艾敬转团热闹",
    "3-上-U1": "晨绒球汉艳服装扮读静停粗影",
    "3-上-U2": "铺水泥紧院印排列规则乱棕迟",
    "3-下-U1": "鸳鸯惠崇芦芽短梅溪泛减",
    "3-下-U2": "守株待兔宋耕释冀守株待兔",
    "4-上-U1": "据堤阔盼滚顿逐渐犹崩震余",
    "4-上-U2": "豌按舒适恐僵硬枪耐探愉曾",
    "4-下-U1": "杂稀篱蜻蜓蝶宿徐疏",
    "4-下-U2": "怒吼脂拭划晌辣挣刷测详",
    "5-上-U1": "宜鹤嫌朱嵌框匣哨恩韵眸",
    "5-上-U2": "汛挽隔懒稳衡协绰似",
    "5-下-U1": "昼耘桑晓蝴蝶蚂蚱樱拔瞎",
    "5-下-U2": "忌瑟都督惩罚遮私寨擂呐",
    "6-上-U1": "德辅喧瀑按巷俯吻羞",
    "6-上-U2": "斑斓精湛绚丽屹立磅礴",
    "6-下-U1": "醋饺摊拌眨宵燃戚贩",
    "6-下-U2": "腊粥跪褐缸筷浓酱搅"
}

In [84]:
char_map = {}
for key, words in unit_index.items():
    # 拆分 key，例如 "1-上-U1" 拆为 ["1-上", "U1"]
    parts = key.rsplit('-', 1) 
    grade_term = parts[0]
    unit_name = parts[1]
    
    for char in words:
        char_map[char] = {'Grade': grade_term, 'Unit': unit_name}
char_map

{'天': {'Grade': '1-上', 'Unit': '识字1'},
 '地': {'Grade': '1-上', 'Unit': '识字1'},
 '人': {'Grade': '1-上', 'Unit': 'U4'},
 '你': {'Grade': '1-上', 'Unit': '识字1'},
 '我': {'Grade': '1-上', 'Unit': '识字1'},
 '他': {'Grade': '1-上', 'Unit': '识字1'},
 '金': {'Grade': '1-上', 'Unit': '识字2'},
 '木': {'Grade': '1-上', 'Unit': '识字2'},
 '水': {'Grade': '3-上', 'Unit': 'U2'},
 '火': {'Grade': '1-上', 'Unit': '识字2'},
 '土': {'Grade': '1-上', 'Unit': '识字2'},
 '子': {'Grade': '1-上', 'Unit': 'U4'},
 '大': {'Grade': '1-上', 'Unit': 'U4'},
 '月': {'Grade': '1-上', 'Unit': 'U4'},
 '儿': {'Grade': '1-上', 'Unit': 'U4'},
 '头': {'Grade': '1-上', 'Unit': 'U4'},
 '里': {'Grade': '1-上', 'Unit': 'U4'},
 '可': {'Grade': '1-上', 'Unit': 'U4'},
 '东': {'Grade': '1-上', 'Unit': 'U4'},
 '西': {'Grade': '1-上', 'Unit': 'U4'},
 '去': {'Grade': '1-上', 'Unit': 'U5'},
 '来': {'Grade': '1-上', 'Unit': 'U5'},
 '不': {'Grade': '1-上', 'Unit': 'U5'},
 '小': {'Grade': '1-上', 'Unit': 'U5'},
 '少': {'Grade': '1-上', 'Unit': 'U5'},
 '牛': {'Grade': '1-上', 'Unit': 'U5'},
 '果

In [85]:
df[['Grade', 'Unit']] = df['char'].apply(lambda x: pd.Series(char_map.get(x, {'Grade': '无', 'Unit': '无'})))
df.head(3)

,id,char,py,wubi,bushou,bihua,pinyin,bishun,yisi,type,Grade,Unit
0,1,卝,guan,None,难检字,4,guàn,2121,卝 guàn 古代儿童将头发束成两角的样子。 卝 kuàng 古同“矿”。,生僻字,无,无
1,2,羋,mi,None,难检字,8,mǐ,12121112,羋 mǐ 同“芈”。,生僻字,无,无
2,3,羐,ling,None,难检字,10,líng,2121121354,羐 líng 古同“蔆”，即“菱”。,生僻字,无,无


In [96]:
df[df['char']=='天']

,id,char,pinyin,wubi,bushou,bihua,py,bishun,yisi,type,Grade,Unit
3313,3314,天,tian,gdi,大,4,tiān,1134,天 tiān 在地面以上的高空：天空。天际。天罡（北斗星）。天渊（上天和深渊，喻差别大）。天...,常用字,1-上,识字1


In [92]:
df.columns=['id', 'char', 'pinyin', 'wubi', 'bushou', 'bihua', 'py', 'bishun',
       'yisi', 'type', 'Grade', 'Unit']

In [93]:

import json
import os

# 1. 创建输出目录（防止文件乱丢）
output_dir = 'hanzi_types'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. 预处理数据：填充空值，确保字段完整性
df_filled = df.fillna({
    'Grade': '未知',
    'Unit': '未知',
    'yisi': '',
    'type': '未分类'
})

# 3. 按照 'type' 列进行分组拆分
# group_name 是类别名称（如'常用字'），group_df 是属于该类别的子集
for group_name, group_df in df_filled.groupby('type'):
    
    # 将该组转换为字典列表
    dict_data = group_df.to_dict(orient='records')
    
    # 构造安全的文件名（移除可能导致路径错误的特殊字符）
    file_name = f"{group_name}.json".replace("/", "_")
    file_path = os.path.join(output_dir, file_name)
    
    # 写入 JSON
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(dict_data, f, ensure_ascii=False, indent=2)
    
    print(f"已生成：{file_path}，包含 {len(dict_data)} 个汉字")

print("\n--- 所有分类文件拆分完成！ ---")

已生成：hanzi_types\常用字.json，包含 2491 个汉字
已生成：hanzi_types\次常用字.json，包含 997 个汉字
已生成：hanzi_types\生僻字.json，包含 16952 个汉字
已生成：hanzi_types\通用字.json，包含 383 个汉字

--- 所有分类文件拆分完成！ ---


In [94]:

# 1. 设置输出路径
output_dir = 'hanzi_by_grade'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. 预处理：填充空值并确保数据类型为字符串
# 假设 Grade 列的数据格式如 "1-上", "2-下" 或 "一年级上"
df_filled = df.fillna({
    'Grade': '未分类',
    'Unit': '未知',
    'type': '普通'
})

# 3. 按 Grade 分组并导出
# groupby 会自动识别 Grade 列中的所有唯一值
for grade_name, group_df in df_filled.groupby('Grade'):
    
    # 清理文件名：将非法字符替换为下划线，防止 Windows/Linux 系统报错
    safe_grade_name = re.sub(r'[\\/*?:"<>|]', '_', str(grade_name))
    
    # 转换为 JSON 格式 (对象数组)
    dict_data = group_df.to_dict(orient='records')
    
    # 构造完整路径，例如: hanzi_by_grade/1-上.json
    file_path = os.path.join(output_dir, f"{safe_grade_name}.json")
    
    # 写入文件
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(dict_data, f, ensure_ascii=False, indent=2)
    
    print(f"成功导出: {file_path} (共 {len(dict_data)} 个生字)")

print("\n✨ 所有年级文件已拆分完毕！")

成功导出: hanzi_by_grade\1-上.json (共 30 个生字)
成功导出: hanzi_by_grade\1-下.json (共 29 个生字)
成功导出: hanzi_by_grade\2-上.json (共 30 个生字)
成功导出: hanzi_by_grade\2-下.json (共 53 个生字)
成功导出: hanzi_by_grade\3-上.json (共 26 个生字)
成功导出: hanzi_by_grade\3-下.json (共 19 个生字)
成功导出: hanzi_by_grade\4-上.json (共 23 个生字)
成功导出: hanzi_by_grade\4-下.json (共 19 个生字)
成功导出: hanzi_by_grade\5-上.json (共 20 个生字)
成功导出: hanzi_by_grade\5-下.json (共 22 个生字)
成功导出: hanzi_by_grade\6-上.json (共 19 个生字)
成功导出: hanzi_by_grade\6-下.json (共 18 个生字)
成功导出: hanzi_by_grade\无.json (共 20515 个生字)

✨ 所有年级文件已拆分完毕！


In [97]:
my_words=list(df.char)

In [98]:
my_words[:10]

['卝', '羋', '羐', '瑴', '彛', '龜', '彞', '龞', '丨', '丩']

In [101]:
my_words=list(df[df['type']=='常用字'].char)

In [ ]:
import os
import requests
import json

# 1. 准备你的 3000 字列表 (假设你已经有了这个 list)
# my_words = ["天", "地", "人", ...] 
#my_words = list("天地人你我他") # 这里替换成你之前的 3000 字列表

# 2. 创建本地存放目录
save_dir = "./data/strokes"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 3. 循环从别人的 GitHub 下载对应的字
base_url = "https://raw.githubusercontent.com/chanind/hanzi-writer-data/master/data/{}.json"


print("开始批量下载笔顺数据...")
for char in my_words:
    file_path = os.path.join(save_dir, f"{char}.json")
    
    # 如果本地已经有了就跳过，省时间
    if os.path.exists(file_path):
        continue
        
    try:
        response = requests.get(base_url.format(char))
        if response.status_code == 200:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(response.text)
            print(f"✅ 已保存: {char}")
        else:
            print(f"❌ 未找到: {char}")
    except Exception as e:
        print(f"⚠️ 下载 {char} 出错: {e}")

print("全部下载完成！现在你可以把 data 文件夹整个推送到你的 GitHub 了。")

开始批量下载笔顺数据...
⚠️ 下载 丰 出错: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
✅ 已保存: 中
✅ 已保存: 卡
✅ 已保存: 串
✅ 已保存: 临
✅ 已保存: 事
⚠️ 下载 乃 出错: HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /chanind/hanzi-writer-data/master/data/%E4%B9%83.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000017F428725A0>, 'Connection to raw.githubusercontent.com timed out. (connect timeout=None)'))
✅ 已保存: 九
⚠️ 下载 久 出错: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
✅ 已保存: 么
⚠️ 下载 丸 出错: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
✅ 已保存: 乏
⚠️ 下载 乌 出错: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
✅ 已保存: 丹
✅ 已保存: 册
✅ 已保存: 乎
✅ 已保存: 乐
✅ 已保存: 丘
✅ 已保存: 乓
✅ 已保存: 乒
✅ 已保存: 乔
✅ 已保存: 丢
✅ 已保存: 乖
✅ 已保存: 乘
✅ 已保存: 了
✅ 已保存: 予
✅ 已保存: 书
✅ 已保存: 一
✅ 已保存: 丁
✅ 已保存: 七
✅ 已